# Part 3 — Coding the Mastermind Solver

This notebook walks through the four milestones of Part 3, building a complete entropy-optimal Mastermind solver step by step.

**Parameters:** $c = 6$ colours, $k = 4$ tokens $\rightarrow$ $6^4 = 1296$ candidate codes.


---
## Milestone 1 — Represent the game

We define the full code space and implement `get_response(secret, guess)`, which returns a `(black, white)` tuple.

**Key point:** when counting white pegs, do *not* reuse positions already matched as black.


In [1]:
import itertools
import math
from collections import defaultdict

COLORS = [1, 2, 3, 4, 5, 6]
CODE_LENGTH = 4

def all_codes():
    """Return the list of all possible codes."""
    return list(itertools.product(COLORS, repeat=CODE_LENGTH))

def get_response(secret, guess):
    """
    Compute the Mastermind response for a (secret, guess) pair.
    Returns (black_pegs, white_pegs).
    """
    # Black pegs: exact position matches
    black = sum(s == g for s, g in zip(secret, guess))

    # White pegs: correct colour but wrong position.
    # Work only on the unmatched positions to avoid double-counting.
    secret_remaining = [s for s, g in zip(secret, guess) if s != g]
    guess_remaining  = [g for s, g in zip(secret, guess) if s != g]

    white = 0
    for g in guess_remaining:
        if g in secret_remaining:
            white += 1
            secret_remaining.remove(g)   # consume the match

    return (black, white)


### Tests

In [2]:
test_cases = [
    ((1, 2, 3, 4), (1, 2, 3, 4), (4, 0)),
    ((1, 2, 3, 4), (4, 3, 2, 1), (0, 4)),
    ((1, 2, 3, 4), (1, 3, 2, 4), (2, 2)),
    ((1, 1, 1, 1), (1, 2, 3, 4), (1, 0)),
    ((1, 2, 1, 2), (2, 1, 2, 1), (0, 4)),
]

all_pass = True
for secret, guess, expected in test_cases:
    result = get_response(secret, guess)
    status = "✓" if result == expected else f"✗  (got {result})"
    print(f"secret={secret}  guess={guess}  expected={expected}  {status}")
    if result != expected:
        all_pass = False

print()
print("All tests passed!" if all_pass else "Some tests FAILED — check get_response.")


secret=(1, 2, 3, 4)  guess=(1, 2, 3, 4)  expected=(4, 0)  ✓
secret=(1, 2, 3, 4)  guess=(4, 3, 2, 1)  expected=(0, 4)  ✓
secret=(1, 2, 3, 4)  guess=(1, 3, 2, 4)  expected=(2, 2)  ✓
secret=(1, 1, 1, 1)  guess=(1, 2, 3, 4)  expected=(1, 0)  ✓
secret=(1, 2, 1, 2)  guess=(2, 1, 2, 1)  expected=(0, 4)  ✓

All tests passed!


---
## Milestone 2 — Partition the candidate set

`partition` groups the current candidates by the response they would each produce to a given guess.
Candidates in the same group are indistinguishable from one another after this guess.


In [ ]:
def partition(candidates, guess):
    """
    Partition candidates by the response each would give to guess.
    Returns a dict mapping response -> list of candidates.
    """
    groups = defaultdict(list)
    for c in candidates:
        groups[get_response(c, guess)].append(c)
    return dict(groups)

### Test — partition all 1296 codes by guess `(1, 1, 2, 2)`

In [4]:
codes = all_codes()
test_partition = partition(codes, (1, 1, 2, 2))

total = sum(len(g) for g in test_partition.values())
print(f"Number of distinct responses: {len(test_partition)}")
print(f"Total candidates across all groups: {total}  (expected 1296)")
print()
print(f"{'Response':<12} {'Count':>6}")
print("-" * 20)
for response, group in sorted(test_partition.items()):
    print(f"{str(response):<12} {len(group):>6}")


Number of distinct responses: 13
Total candidates across all groups: 1296  (expected 1296)

Response      Count
--------------------
(0, 0)          256
(0, 1)          256
(0, 2)           96
(0, 3)           16
(0, 4)            1
(1, 0)          256
(1, 1)          208
(1, 2)           36
(2, 0)          114
(2, 1)           32
(2, 2)            4
(3, 0)           20
(4, 0)            1


---
## Milestone 3 — Expected entropy and best guess

### Reflection 3.1 — Are the entropy of the response and the expected remaining entropy the same?

After a guess, each response $r$ leaves a group $S_r$ of surviving candidates.
Treating survivors as equally likely, the remaining entropy after observing $r$ is $\log_2|S_r|$.

The **expected remaining entropy** is:
$$\mathbb{E}[H_{\text{remaining}}] = \sum_r p(r)\,\log_2|S_r|$$

Using $p(r) = |S_r|/n$, where $n = |\text{candidates}|$:
$$= \sum_r \frac{|S_r|}{n}\Bigl(\log_2 n + \log_2 p(r)\Bigr) = \log_2 n + \sum_r p(r)\log_2 p(r) = \log_2 n - H(\text{response})$$

Since $\log_2 n$ is a constant for the current turn, **maximising the entropy of the response distribution is exactly equivalent to minimising the expected remaining entropy**.  
The function below computes $H(\text{response}) = -\sum_r p(r)\log_2 p(r)$, which is therefore the correct objective.


In [5]:
def expected_entropy(candidates, guess):
    """
    Compute the entropy of the response distribution induced by guess
    on the current candidate set.
    By Reflection 3.1, maximising this minimises expected remaining entropy.
    """
    parts = partition(candidates, guess)
    total = len(candidates)
    h = 0.0
    for group in parts.values():
        p = len(group) / total
        if p > 0:
            h -= p * math.log2(p)
    return h


def best_guess(candidates, all_codes_list):
    """
    Return the guess from all_codes_list that maximises expected_entropy.
    Ties are broken in favour of guesses that are still valid candidates
    (so we solve the game immediately when possible).
    """
    best_h = -1
    best_g = None
    cand_set = set(candidates)
    for g in all_codes_list:
        h = expected_entropy(candidates, g)
        if h > best_h or (h == best_h and g in cand_set and best_g not in cand_set):
            best_h = h
            best_g = g
    return best_g


### Sanity check — optimal first guess

In [6]:
codes = all_codes()

first_entropies = [(expected_entropy(codes, g), g) for g in codes]
best_h = max(h for h, _ in first_entropies)
optimal_firsts = [g for h, g in first_entropies if abs(h - best_h) < 1e-9]

print(f"Best first-guess response entropy : {best_h:.4f} bits")
print(f"Number of tied optimal first guesses: {len(optimal_firsts)}")
print(f"Examples: {optimal_firsts[:6]}")
print()
all_4_distinct = all(len(set(g)) == 4 for g in optimal_firsts)
print(f"All optimal first guesses use exactly 4 distinct colours: {all_4_distinct}")


Best first-guess response entropy : 3.0567 bits
Number of tied optimal first guesses: 360
Examples: [(1, 2, 3, 4), (1, 2, 3, 5), (1, 2, 3, 6), (1, 2, 4, 3), (1, 2, 4, 5), (1, 2, 4, 6)]

All optimal first guesses use exactly 4 distinct colours: True


---
## Milestone 4 — The interactive solver

The cell below defines the full interactive loop.
The solver picks the entropy-optimal guess each turn; you report the response as `black white` (e.g. `2 1`).


In [ ]:
def solve():
    """Interactive Mastermind solver. Run this cell and follow the prompts."""
    candidates = all_codes()
    codes = all_codes()
    turn = 1

    print(f"I will guess your secret code ({CODE_LENGTH} pegs, colours {COLORS}).")
    print("After each guess, enter the response as: black white  (e.g. '2 1')\n")

    while True:
        if len(candidates) == 1:
            print(f"Turn {turn}: The secret must be {candidates[0]}!")
            break

        guess = best_guess(candidates, codes)
        print(f"Turn {turn}: I guess {guess}")

        response_str = input("Your response (black white): ")
        black, white = map(int, response_str.split())

        if black == CODE_LENGTH:
            print(f"Solved in {turn} turn(s)!")
            break

        candidates = [c for c in candidates if get_response(c, guess) == (black, white)]
        print(f"  {len(candidates)} candidate(s) remaining.\n")
        turn += 1

# Uncomment the line below to play interactively:
solve()


I will guess your secret code (4 pegs, colours [1, 2, 3, 4, 5, 6]).
After each guess, enter the response as: black white  (e.g. '2 1')

Turn 1: I guess (1, 2, 3, 4)


Your response (black white):  1 1


  252 candidate(s) remaining.

Turn 2: I guess (2, 5, 3, 6)


Your response (black white):  0 1


  31 candidate(s) remaining.

Turn 3: I guess (1, 4, 5, 1)


### Non-interactive demo

To verify correctness without manual input, we run the solver automatically against a known secret.


In [9]:
def solve_auto(secret):
    """Run the solver non-interactively against a known secret."""
    candidates = all_codes()
    codes = all_codes()
    history = []

    for turn in range(1, 20):
        guess = candidates[0] if len(candidates) == 1 else best_guess(candidates, codes)
        response = get_response(secret, guess)
        history.append((turn, guess, response, len(candidates)))
        if response == (CODE_LENGTH, 0):
            break
        candidates = [c for c in candidates if get_response(c, guess) == response]

    return history

# Demo: solve for a specific secret
secret = (3, 5, 2, 6)
print(f"Solving for secret: {secret}\n")
print(f"{'Turn':<6} {'Guess':<20} {'Response':<12} {'Candidates left'}")
print("-" * 56)
for turn, guess, response, n_cands in solve_auto(secret):
    note = "  ← solved!" if response == (CODE_LENGTH, 0) else ""
    print(f"{turn:<6} {str(guess):<20} {str(response):<12} {n_cands}{note}")


Solving for secret: (3, 5, 2, 6)

Turn   Guess                Response     Candidates left
--------------------------------------------------------
1      (1, 2, 3, 4)         (0, 2)       1296
2      (2, 4, 5, 6)         (1, 2)       312
3      (5, 4, 6, 3)         (0, 3)       34
4      (3, 5, 2, 6)         (4, 0)       5  ← solved!
